In [4]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
from diskcache import Cache
from itertools import chain
from collections import Counter
import contextlib
from unidecode import unidecode
from nameparser import HumanName
from json_repair import repair_json

from dataclasses import dataclass
from itertools import chain

import igraph as ig
import matplotlib.pyplot as plt
import seaborn.objects as so
import seaborn as sns

from utils.pandas_setup import pandas_setup
pandas_setup()

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

def normalise_name(in_name: str='') -> list:
    # print(f'{in_name = }')
    in_name = ' '.join([part.strip() for part in unidecode(in_name).split(' ')])
    in_name = in_name.title()
    name = HumanName(in_name)
    
    if name.last == 'Athey':
        name.first = 'Susan.'
    if name.last == 'Deaton':
        name.first = 'Angas.'

    if name.middle == "":
        fullname = f'{name.first} {name.last}'
    else:
        fullname = f'{name.first} {name.middle} {name.last}'

    if fullname == 'David A Hensher':
        fullname = 'David A. Hensher'
    if fullname == 'Ja Robinson':
        fullname = 'James A. Robinson'
    if fullname == "Nicola Fuchs-Schuendeln":
        fullname = "Nicola Fuchs-Schundeln"
    if fullname == "Georg Weizsaecker":
        fullname = "Georg Weizsacker"
    if fullname == "Che Yeon-Koo":
        fullname = "Yeon-Koo Che"
    if fullname == "Dong-Yang Zhang":
        fullname = "Dongyang Zhang"

    # return {'first': name.first, 'middle': name.middle, 'family': name.last, 'fullname': fullname}
    return [name.first, name.middle, name.last, fullname]


def dummy(input: str='') -> dict:
    return {'key1': 1, 'key2': 2}

In [ ]:
MY_DATA_PATH = Path('/home/lc/m/working/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/economicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')
MY_BACKUP_FILE = Path(MY_DATA_PATH / 'econ_backup.duckdb')

class SetUp:

    def __init__(self):
        self._open_db()
        self._open_cache()
        return

    def _open_db(self):
        self.db = duckdb.connect()
        self.db.sql(f"ATTACH IF NOT EXISTS '{str(MY_DATABASE_FILE)}' AS project")
        self.db.sql(f"ATTACH IF NOT EXISTS '{str(MY_BACKUP_FILE)}' AS backup")
        self.db.sql("ATTACH IF NOT EXISTS '/home/lc/m/openalex_june25/authors.duckdb' AS authors")
        self.db.sql("ATTACH IF NOT EXISTS '/home/lc/m/openalex_june25/sources.duckdb' AS sources")
        self.db.sql("ATTACH IF NOT EXISTS '/home/lc/m/openalex_june25/institutions.duckdb' AS institutions")
        self.db.sql("ATTACH IF NOT EXISTS '/home/lc/m/openalex_dec24/duckdb/works.duckdb' AS works")
        # self.db.sql("ATTACH IF NOT EXISTS '/home/lc/m/openalex_dec24/duckdb/institutions.duckdb'")
        # self.db.sql("ATTACH IF NOT EXISTS '/home/lc/m/working/orcid.duckdb'")
        self.db.sql(""" SET memory_limit = '24GB';
                        SET threads = 2;
                        SET preserve_insertion_order = false;
                        SET order_by_non_integer_literal=true;
                        SET enable_progress_bar = false;
                        SET temp_directory = '/home/lc/m/.tmp';
                    """)
        for tab in ['project.edge_list_sources', 'project.edge_list_institutions', 'project.edge_list_both']: 
            self.db.sql(f"DROP TABLE IF EXISTS {tab}")

        # with contextlib.suppress(Exception):
        #     self.db.create_function('normalise_name', 
        #                                 normalise_name, 
        #                                 return_type=duckdb.duckdb.typing.DuckDBPyType(dict[str, str]), 
        #                                 exception_handling='return_null',
        #                                 null_handling='special',
        #                                 side_effects=True
        #                             )
                        
        self.db.sql("SHOW ALL TABLES").show()
        return
    
    def _open_cache(self):
        self.cache = Cache(MY_CACHE_FILE, size_limit=int(1e11))
        # self.cache.clear()
        print(f'{self.cache.check() = }')
        print(f'{self.cache.volume() = }')        
        return
    
    def _cache_manager(self, task=None):
        # Check cache
        result = self.cache.get(task)
        # if 'S170166683' in task:
        #     result = False
        # print(f'_cache_manager CHECK {type(result) = }')
        if isinstance(result, pd.DataFrame):
            print("\rC", end='\r')
            return result
        if isinstance(result, list):
            print("\rC", end='\r')
            self.cache[task] = pd.json_normalize(result, max_level=1)
            return self.cache[task]
        if result == "__NONE__":
            print(f"\rC with __NONE__ returned for {task = }", end='\r')
            return None

        # Not in cache, go to Web
        try:
            result = self._extractor(task=task)
            # print(f'_cache_manager WEB {type(result) = } {len(result) = }')
            if isinstance(result, (list, str)):
                print(f"\rW {len(result) = }", end='\r')
                self.cache[task] = pd.json_normalize(result, max_level=1)
                # print(result)
                # print(self.cache[task])
                return self.cache[task]
            if result is None:
                print("\rW", end='\r')
                self.cache[task] = "__NONE__"
                return None
        except Exception as e:        # print(self.db.sql("DESCRIBE TABLE project.raw").df())
            print(f'_cache_manager failed to read WEB\n{e = }\n{result = }\n{task = }')
        return None
        
    def _extractor(self, task=None):
        # print(f'{task = }')
        if 'autocomplete' in task: # type: ignore # or 'Authors' in task:
            return eval(task) # type: ignore
        query = f'{task}.paginate(per_page=200)'
        return list(chain(*eval(query)))
    
    def _ensure_proper_json(self, response):
        print('repair_jason called')
        response_repaired = []
        response_repaired.extend(repair_json(row) for row in response)
        return response_repaired
    
    def name_of_global_obj(self, obj=None):
        for objname, oid in globals().items():
            if oid is obj:
                return objname

In [ ]:
# s = SetUp()
# s.db.close()

┌──────────────┬─────────┬─────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────